# Install the Package
Here we're installing it directly from GitHub while it's in development.

In [1]:
conda env list


# conda environments:
#
base                   D:\software\Programs\miniconda3
hyocr-cpu              D:\software\Python\envs\hyocr-cpu
inno_optical_predict   D:\software\Python\envs\inno_optical_predict
nanobot                D:\software\Python\envs\nanobot
py312                  D:\software\Python\envs\py312
vanna                * D:\software\Python\envs\vanna


Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install "vanna[fastapi]"

  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
Using cached starlette-1.3.1-py3-none-any.whl (73 kB)

   ---------- ----------------------------- 1/4 [uvicorn]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ---------------------------------------- 4/4 [fastapi]



In [5]:
!pip install "vanna[flask,openai]"

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.16.0-cp312-cp312-win_amd64.whl.metadata (5.3

# Download a Sample Database

In [6]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# Imports

In [9]:
from vanna import Agent, AgentConfig
from vanna.servers.fastapi import VannaFastAPIServer
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
# from vanna.integrations.llm.anthropic import AnthropicLlmService
from vanna.integrations.llm.openai import OpenAILlmService
from vanna.tools import RunSqlTool, VisualizeDataTool
from vanna.integrations.databases.relational.sqlite import SqliteRunner
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool
from vanna.integrations.local.agent_memory import DemoAgentMemory
from vanna.capabilities.sql_runner import RunSqlToolArgs
from vanna.tools.visualize_data import VisualizeDataArgs

# Define your User Authentication
Here we're going to say that if you're logged in as `admin@example.com` you get the `admin1` user id, otherwise you get the `user1` user id

In [10]:
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        # In production, validate cookies/JWTs here
        user_email = request_context.get_cookie('vanna_email')
        if not user_email:
            raise ValueError("Missing 'vanna_email' cookie for user identification")
        
        print(f"Resolving user for email: {user_email}")

        if user_email == "admin@example.com":
            return User(id="admin1", email=user_email)
        
        return User(id="user1", email=user_email)

# Define the Tools

In [11]:
tools = ToolRegistry()
tools.register(RunSqlTool(sql_runner=SqliteRunner(database_path="./Chinook.sqlite")))
tools.register(VisualizeDataTool())
agent_memory = DemoAgentMemory(max_items=1000)
tools.register(SaveQuestionToolArgsTool())
tools.register(SearchSavedCorrectToolUsesTool())

In [ ]:
# Set up LLM
llm = OpenAILlmService(model="Qwen/Qwen3.5-27B", api_key="sk-Hu-lzpN2fccQ_th4u6AkBw", base_url="https://aihub.innolight.com:50443/v1")

# Create agent with your options
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    config=AgentConfig(),
    agent_memory=agent_memory
)

# 4. Create and run server
server = VannaFastAPIServer(agent)
server.run()

In [ ]:
# # Set up LLM
# llm = AnthropicLlmService(model="claude-sonnet-4-5", api_key="sk-ant-...")

# # Create agent with your options
# agent = Agent(
#     llm_service=llm,
#     tool_registry=tools,
#     user_resolver=SimpleUserResolver(),
#     config=AgentConfig(),
#     agent_memory=agent_memory
# )

# # 4. Create and run server
# server = VannaFastAPIServer(agent)
# server.run()